Install R and required packages

In [1]:
# install rpy2 to run R inside Colab
!pip install rpy2 --quiet

Activate R environment

In [2]:
%load_ext rpy2.ipython

Load libraries in R

In [3]:
%%R

install.packages("sqldf")
install.packages("dplyr")

library(sqldf)
library(dplyr)

print("Libraries loaded successfully")

[1] "Libraries loaded successfully"


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’

trying URL 'https://cran.rstudio.com/src/contrib/gsubfn_0.7.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/proto_1.0.0.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/RSQLite_2.4.6.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/chron_2.3-62.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/sqldf_0.4-12.tar.gz'

The downloaded source packages are in
	‘/tmp/Rtmp0DrfCk/downloaded_packages’
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/dplyr_1.2.0.tar.gz'
Content type 'application/x-gzip' length 922393 bytes (900 KB)
downloaded 900 KB


The downloaded source packages are in
	‘/tmp/Rtmp0DrfCk/downloaded_packages’
Loading required package: gsubfn
Loading required package: proto
Loading required package: RSQLite

Attaching

Upload datasets

In [4]:
from google.colab import files
uploaded = files.upload()

Saving app_events.csv to app_events.csv
Saving complaints.csv to complaints.csv
Saving customers.csv to customers.csv
Saving data_dictionary.csv to data_dictionary.csv
Saving deliveries.csv to deliveries.csv
Saving drivers.csv to drivers.csv
Saving hubs.csv to hubs.csv
Saving incidents.csv to incidents.csv
Saving orders.csv to orders.csv
Saving vehicles.csv to vehicles.csv


Load datasets into R

In [13]:
%%R

app_event <- read.csv("/content/app_events.csv")
complaint <- read.csv("/content/complaints.csv")
customer <- read.csv("/content/customers.csv")
drivers <- read.csv("/content/drivers.csv")
hubs <- read.csv("/content/hubs.csv")
incidents <- read.csv("/content/incidents.csv")
orders <- read.csv("/content/orders.csv")
vehicle <- read.csv("/content/vehicles.csv")

print("Datasets loaded in R")

[1] "Datasets loaded in R"


# Query 1 – Order volume by service type

Business problem

Management wants to understand which service category generates the highest operational demand.

In [14]:
%%R

query1 <- sqldf("
SELECT service_type,
COUNT(order_id) AS total_orders,
ROUND(AVG(order_value),2) AS avg_order_value
FROM orders
GROUP BY service_type
ORDER BY total_orders DESC
")

print(query1)

  service_type total_orders avg_order_value
1    Passenger          341           96.07
2       Parcel          308           87.62
3       Retail          297           90.01
4     Business          165           92.25
5      Medical          139           87.14


Interpretation

Shows which services (Passenger, Parcel, Retail) generate most orders and highest revenue.

# Query 2 – High priority orders


Business problem

High priority services may require more resources and faster response times.

In [15]:
%%R

query2 <- sqldf("
SELECT priority_level,
COUNT(*) AS total_orders
FROM orders
GROUP BY priority_level
")

print(query2)

  priority_level total_orders
1       Critical           91
2           High          308
3            Low          348
4         Medium          503


Interpretation

Identifies workload distribution by priority level.

# Query 3 – Customer complaints by type

Business problem

Management needs to identify the most frequent service failures.

In [16]:
%%R

query3 <- sqldf("
SELECT complaint_type,
COUNT(*) AS total_complaints,
ROUND(AVG(compensation_amount),2) AS avg_compensation
FROM complaint
GROUP BY complaint_type
ORDER BY total_complaints DESC
")

print(query3)

     complaint_type total_complaints avg_compensation
1             Delay              101            18.05
2      MissedPickup               64            22.59
3          AppIssue               53            19.61
4   DriverBehaviour               51            21.15
5 SupportExperience               20            17.13
6           Billing               16            23.87
7            Damage               15            23.98


Interpretation

Highlights main sources of dissatisfaction and financial compensation impact.

# Query 4 – Orders with complaints

Business problem

Determine how many orders result in customer dissatisfaction.

In [17]:
%%R

query4 <- sqldf("
SELECT o.order_id,
o.service_type,
o.order_value,
c.complaint_type,
c.severity
FROM orders o
LEFT JOIN complaint c
ON o.order_id = c.order_id
WHERE c.complaint_id IS NOT NULL
")

print(head(query4))

  order_id service_type order_value complaint_type severity
1   O00814    Passenger       27.90       AppIssue     High
2   O00628    Passenger       52.85   MissedPickup   Medium
3   O00384      Medical       12.58          Delay     High
4   O00406       Retail       59.17          Delay   Medium
5   O00154     Business      105.88          Delay   Medium
6   O00147       Retail       32.27          Delay   Medium


Interpretation

Links operational transactions to customer complaints.

# Query 5 – Customer engagement analysis

Business problem

Understand whether highly engaged customers generate more orders.

In [18]:
%%R

query5 <- sqldf("
SELECT customer_type,
ROUND(AVG(loyalty_score),2) AS avg_loyalty,
ROUND(AVG(app_engagement_score),2) AS avg_engagement
FROM customer
GROUP BY customer_type
")

print(query5)

  customer_type avg_loyalty avg_engagement
1      Consumer       60.32          58.56
2    Enterprise       59.33          55.17
3           SME       57.35          57.69


Interpretation

Shows relationship between engagement and customer category.

# Query 6 – Driver performance analysis

Business problem

Driver performance may influence service quality and delays.

In [19]:
%%R

query6 <- sqldf("
SELECT base_zone,
COUNT(driver_id) AS total_drivers,
ROUND(AVG(driver_rating),2) AS avg_rating,
ROUND(AVG(training_score),2) AS avg_training
FROM drivers
GROUP BY base_zone
ORDER BY avg_rating DESC
")

print(query6)

   base_zone total_drivers avg_rating avg_training
1  RiverSide            10       4.33        73.29
2      South            21       4.28        71.83
3    CENTRAL            10       4.26        62.37
4      north            11       4.25        72.63
5        Ctr             6       4.25        63.52
6    Airport             9       4.25        80.97
7       East            14       4.22        76.70
8      SOUTH             8       4.20        81.46
9    AIRPORT            10       4.18        80.65
10      West            10       4.14        76.03
11     NORTH            12       4.14        78.15
12   Central            12       4.14        72.56
13      WEST            10       4.09        78.89
14 Riverside             7       4.09        68.99
15     North            13       3.90        79.69
16      EAST             7       3.90        79.43


Interpretation

Identifies zones with strongest or weakest driver performance.

# Query 7 – Vehicle maintenance risk

Business problem

Poor vehicle condition may cause service delays.

In [20]:
%%R

query7 <- sqldf("
SELECT vehicle_type,
COUNT(vehicle_id) AS total_vehicles,
ROUND(AVG(battery_health_pct),2) AS avg_battery_health
FROM vehicle
GROUP BY vehicle_type
")

print(query7)

  vehicle_type total_vehicles avg_battery_health
1     CargoVan             30              73.38
2       Diesel             19              70.56
3           EV             43              82.12
4       Hybrid             28              76.84


Interpretation

Shows fleet condition across vehicle types.

# Query 8 – Incident frequency

Business problem

Operational disruptions affect service reliability.

In [21]:
%%R

query8 <- sqldf("
SELECT incident_type,
COUNT(*) AS incident_count,
ROUND(AVG(resolved_hours),2) AS avg_resolution_time
FROM incidents
GROUP BY incident_type
ORDER BY incident_count DESC
")

print(query8)

     incident_type incident_count avg_resolution_time
1     ProofMissing             46               10.77
2   CustomerNoShow             44               13.89
3   RouteDeviation             43               13.73
4     VehicleFault             37                9.15
5     BatteryAlert             36               11.71
6     AppSyncError             31               12.66
7 TemperatureIssue             29               12.92
8   SafetyNearMiss             14                9.67


In [ ]:
Interpretation

Highlights most common operational disruptions.

# Query 9 – Orders by zone

Business problem

Identify high-demand zones requiring resource allocation.

In [22]:
%%R

query9 <- sqldf("
SELECT pickup_zone,
COUNT(order_id) AS total_orders,
ROUND(AVG(order_value),2) AS avg_order_value
FROM orders
GROUP BY pickup_zone
ORDER BY total_orders DESC
")

print(query9)

   pickup_zone total_orders avg_order_value
1         East          104           92.22
2        South          103           92.40
3         EAST          103           91.33
4    RiverSide           86           80.38
5      Airport           85          108.85
6         WEST           84           89.20
7          Ctr           80           94.50
8      Central           79           77.20
9      CENTRAL           79           93.58
10       SOUTH           78           88.18
11        West           71           87.18
12   Riverside           65           91.90
13       north           64           96.46
14       NORTH           60           89.35
15     AIRPORT           59           96.74
16       North           50           86.10


Interpretation

Identifies busiest operational zones.

# Query 10 – App performance monitoring

Business problem

Slow API response times may affect user satisfaction.

In [23]:
%%R

query10 <- sqldf("
SELECT device_type,
COUNT(event_id) AS total_events,
ROUND(AVG(api_latency_ms),2) AS avg_latency
FROM app_event
GROUP BY device_type
ORDER BY avg_latency DESC
")

print(query10)

  device_type total_events avg_latency
1         Web           92      474.68
2     Android          315      464.90
3         iOS          233      463.15


Interpretation

Shows platform performance differences across devices.

# Query 11 – Complaint resolution efficiency

Business problem

Slow resolution increases dissatisfaction and costs.

In [24]:
%%R

query11 <- sqldf("
SELECT severity,
ROUND(AVG(resolution_days),2) AS avg_resolution_days,
ROUND(AVG(compensation_amount),2) AS avg_compensation
FROM complaint
GROUP BY severity
")

print(query11)

  severity avg_resolution_days avg_compensation
1     High               13.12            38.85
2      Low                6.56             9.06
3   Medium                6.17            17.37


Interpretation

Shows relationship between severity and resolution effort.